In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd
from scripts.plotting import *
from scripts.denoising import *
from sklearn.preprocessing import StandardScaler
from scipy.linalg import svd
from scipy.spatial.distance import pdist, squareform, jaccard
from scipy.linalg import norm

In [ ]:
X = pd.read_csv("./data/s_curve/uniform/s_curve_noisy_hd_position_matrix.csv")
Y = pd.read_csv("./data/s_curve/uniform/s_curve_noisy_hd_velocity_matrix.csv")
t = pd.read_csv("./data/s_curve/uniform/s_curve_gt_latent_time_vector.csv")
t = list(t["t"])

X_gt = pd.read_csv("./data/s_curve/uniform/s_curve_gt_position_matrix.csv")
Y_gt = pd.read_csv("./data/s_curve/uniform/s_curve_gt_velocity_matrix.csv")

X.shape, Y.shape

In [ ]:
scaler = StandardScaler()
X_std = scaler.fit_transform(X)
pca_elbow_plot(X_std, top_k=30, dot_size=30)

In [ ]:
scaler = StandardScaler()
Y_std = scaler.fit_transform(Y)
pca_elbow_plot(Y_std, top_k=30, dot_size=30)

In [ ]:
# Perform PCA on the high-dimensional noisy position matrix
pca = PCA(n_components=3)
principal_components = pca.fit_transform(X_std)  # Shape: (n_samples, 3)

# Project the high-dimensional velocity matrix using the same PCA
velocity_components = pca.fit_transform(Y_std)  # Shape: (n_samples, 3)

rotation = np.array([[0,0,1],
                  [0,1,0],
                  [1,0,0]])

# Visualize using the provided function
plot_3d_with_quiver(
    principal_components @ rotation,
    velocity_components @ rotation,
    t,
    arrow_size=1,
    title="3D S-Curve with General Derivatives (PCA Projection)",
)

In [ ]:
r = 3
X_hat, Y_hat, V_hat = subspace_denoised(X_std, Y_std, r, alpha=0.5)

In [ ]:
rotation = np.array([[0,0,1],
                  [1,0,0],
                  [0,1,0]])
# Visualize using the provided function
plot_3d_with_quiver(
    X_std @ V_hat.T[:X_hat.shape[1],:r] @ rotation,
    Y_std @ V_hat.T[:X_hat.shape[1],:r] @ rotation,
    t,
    arrow_size=1,
    title="3D S-Curve with General Derivatives (PCA Projection)",
)

In [ ]:
pca_elbow_plot(X_hat, top_k=30, dot_size=30)

In [ ]:
pca_elbow_plot(Y_hat, top_k=30, dot_size=30)

In [ ]:
def cosine_angle(Y):
    cosine_sim = Y @ Y.T
    norms = np.linalg.norm(Y, axis=1)
    cosine_angles = cosine_sim / np.outer(norms, norms)
    return cosine_angles

def spectral_norm(matrix):
    u, s, vh = svd(matrix)
    return np.max(s)

def compute_jaccard_distance(matrix1, matrix2, k=10):
    """Compute Jaccard distance between k-nearest neighbor graphs."""
    dist1 = squareform(pdist(matrix1, metric="euclidean"))
    dist2 = squareform(pdist(matrix2, metric="euclidean"))
    
    # Get indices of k nearest neighbors (excluding self)
    neighbors1 = [set(np.argsort(row)[1:k+1]) for row in dist1]
    neighbors2 = [set(np.argsort(row)[1:k+1]) for row in dist2]

    # Compute Jaccard similarity manually using set operations
    jaccard_similarities = [
        len(n1.intersection(n2)) / len(n1.union(n2)) if len(n1.union(n2)) > 0 else 1
        for n1, n2 in zip(neighbors1, neighbors2)
    ]
    
    # Convert Jaccard similarity to Jaccard distance (1 - similarity)
    jaccard_distances = [1 - sim for sim in jaccard_similarities]
    
    return np.mean(jaccard_distances)

In [ ]:
np.random.seed(42)

r = 3
_, X_sigma_raw, _ = svd(X_std)
X_total_variance_raw = np.sum(X_sigma_raw ** 2)

_, Y_sigma_raw, _ = svd(Y_std)
Y_total_variance_raw = np.sum(Y_sigma_raw ** 2)

Y_cosine = cosine_angle(Y_std)
Y_cosine_gt = cosine_angle(Y_gt)

X_jaccard_gt_raw = compute_jaccard_distance(X_gt, X_std, k=30)  # Ground truth Jaccard distance
Y_cosine_norm_gt_raw = spectral_norm(Y_cosine_gt - Y_cosine)  # Should be zero since it's compared to itself

results = {"X": [], "Y": [], "X_jaccard": [], "Y_cosine_norm": [],
           "X_jaccard_gt": [], "Y_cosine_norm_gt": []}
alphas = np.linspace(0, 1, 11)  # Generate alpha values

# Iterate over alpha values and compute explained variance
for alpha in alphas:
    X_hat, Y_hat, V_hat = subspace_denoised(X_std, Y_std, r, alpha=alpha)
    
    _, X_sigma_hat, _ = svd(X_hat)
    X_total_variance_hat = np.sum(X_sigma_hat ** 2)
    results["X"].append(X_total_variance_hat / X_total_variance_raw)
    
    _, Y_sigma_hat, _ = svd(Y_hat)
    Y_total_variance_hat = np.sum(Y_sigma_hat ** 2)
    results["Y"].append(Y_total_variance_hat / Y_total_variance_raw)
    
    Y_cosine_hat = cosine_angle(Y_hat)
    results["X_jaccard"].append(compute_jaccard_distance(X_std, X_hat, k=30))
    results["Y_cosine_norm"].append(spectral_norm(Y_cosine_hat - Y_cosine))
    
    results["X_jaccard_gt"].append(compute_jaccard_distance(X_gt, X_hat, k=30))
    results["Y_cosine_norm_gt"].append(spectral_norm(Y_cosine_hat - Y_cosine_gt))

    
# Plot explained variance for X and Y on two different Y-axes
fig, ax1 = plt.subplots(figsize=(8, 6))

ax1.set_xlabel('Alpha')
ax1.set_ylabel('Explained Variance (X)', color='tab:blue')
ax1.plot(alphas, results["X"], marker='o', label='X Explained Variance', color='tab:blue')
ax1.tick_params(axis='y', labelcolor='tab:blue')

ax2 = ax1.twinx()  # Create a second y-axis
ax2.set_ylabel('Explained Variance (Y)', color='tab:orange')
ax2.plot(alphas, results["Y"], marker='o', label='Y Explained Variance', color='tab:orange')
ax2.tick_params(axis='y', labelcolor='tab:orange')

fig.suptitle('Explained Variance for X and Y Across Alpha')
fig.tight_layout()
plt.show()

# Plot the average explained variance
average_explained_variance = np.mean([results["X"], results["Y"]], axis=0)
plt.figure(figsize=(8, 6))
plt.plot(alphas, average_explained_variance, marker='o', label='Average Explained Variance', color='tab:green')
plt.xlabel('Alpha')
plt.ylabel('Average Explained Variance')
plt.title('Average Explained Variance Across Alpha')
plt.legend()
plt.grid()
plt.show()

In [ ]:
# Plot Jaccard distance
plt.figure(figsize=(8, 6))
plt.plot(alphas, results["X_jaccard"], marker='o', label='Jaccard Distance', color='tab:red')
plt.xlabel('Alpha')
plt.ylabel('Jaccard Distance')
plt.title('Jaccard Distance of Nearest Neighbors Across Alpha')
plt.legend()
plt.grid()
plt.show()

# Plot Jaccard distance
plt.figure(figsize=(8, 6))
plt.plot(alphas, results["Y_cosine_norm"], marker='o', label='Distance of cosine similarity', color='tab:red')
plt.xlabel('Alpha')
plt.ylabel('2-norm of cosine similarity')
plt.title('Cosine similarity of velocity matrix')
plt.legend()
plt.grid()
plt.show()

In [ ]:
# Plot Jaccard distance with reference line
plt.figure(figsize=(8, 6))
plt.plot(alphas, results["X_jaccard_gt"], marker='o', label='Jaccard Distance', color='tab:red')
plt.axhline(y=X_jaccard_gt_raw, color='black', linestyle='--', label='Jaccard GT Raw')
plt.xlabel('Alpha')
plt.ylabel('Jaccard Distance')
plt.title('Jaccard Distance of Nearest Neighbors Across Alpha')
plt.legend()
plt.grid()
plt.show()

# Plot Cosine Norm with reference line
plt.figure(figsize=(8, 6))
plt.plot(alphas, results["Y_cosine_norm_gt"], marker='o', label='Cosine Norm', color='tab:blue')
plt.axhline(y=Y_cosine_norm_gt_raw, color='black', linestyle='--', label='Cosine Norm GT Raw')
plt.xlabel('Alpha')
plt.ylabel('2-Norm of Cosine Similarity')
plt.title('Cosine Similarity of Velocity Matrix Across Alpha')
plt.legend()
plt.grid()
plt.show()